# 12 实战：从原始文档到 SFT 数据集

前面 11 节是"零件"，这一节把它们装成一条**能重复运行**的数据管线。

### 输入

```
data/docs/          4 个文档（md / txt / html，含转载重复）
data/raw_sft/       FAQ JSON、Alpaca JSONL、对话日志 JSONL
```

### 输出（写到 `python/pandas/output/`）

| 文件 | 内容 |
| --- | --- |
| `pretrain.jsonl` | 继续预训练语料：`{"text": …}` |
| `sft_train.jsonl` / `sft_val.jsonl` | 指令微调数据：Alpaca 格式 |
| `pipeline_records.parquet` | 统一内部表，便于增量更新 |
| `quality_report.csv` | 每一步的条数与丢弃原因 |

### 流程

```
盘点 → 抽取 → 清洗 → 切分 → 去重 → 拼接指令数据 → 质检 → 划分 → 导出 → 验收
```

> 环境：`llm_sft`，从 `python/pandas` 目录启动。运行一遍约 2 秒。

In [1]:
import hashlib
import json
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
from bs4 import BeautifulSoup, Comment

pd.set_option("display.unicode.east_asian_width", True)
pd.set_option("display.max_colwidth", 60)

DATA = Path("data")
if not DATA.exists():
    DATA = Path("python/pandas/data")
OUT = DATA.parent / "output" / "sft_pipeline"     # 本节的产物单独放一个目录，避免和其他小节互相覆盖
OUT.mkdir(parents=True, exist_ok=True)

print("数据目录:", DATA.resolve())
print("输出目录:", OUT.resolve())

数据目录: D:\Repository\Jianyu-is-learning\python\pandas\data
输出目录: D:\Repository\Jianyu-is-learning\python\pandas\output\sft_pipeline


## 0. 盘点：先用一张表看清"有什么"

任何数据项目的第一步都一样：**列出所有输入文件、类型、大小**。
这一步用 `pathlib` + 一行 `DataFrame` 就够。

In [2]:
files = []
for root in [DATA / "docs", DATA / "raw_sft"]:
    for path in sorted(root.rglob("*")):
        if path.is_file():
            files.append({
                "path": path,
                "name": path.name,
                "type": path.suffix.lstrip("."),
                "folder": root.name,
                "kb": round(path.stat().st_size / 1024, 1),
            })
inventory = pd.DataFrame(files)
inventory.groupby(["folder", "type"]).agg(文件数=("name", "count"), 总KB=("kb", "sum"))

文件数  总KB
folder  type               
docs    html        1   1.9
        md          1   1.4
        txt         2   1.8
raw_sft json        1   1.2
        jsonl       2   6.7

## 1. 抽取 + 清洗

规则不多，但每一条都有明确目的（详见第 09 节）：

- HTML 去 `script/style/nav/footer/注释`，只取 `<article>`；
- 零宽字符、全角空格、重复空白清掉；
- 页码行、重复页眉删掉；
- ```` ```代码块``` ```` 先保护、后还原，避免缩进被压平。

In [3]:
CODE_BLOCK = re.compile(r"```.*?```", re.S)
PAGE_NUMBER = re.compile(r"^[—–\-]{0,2}\s*\d{1,3}\s*[—–\-]{0,2}$")
HEADING_MARK = re.compile(r"^#{1,6}\s+")


def extract_text(path: Path) -> str:
    if path.suffix == ".html":
        soup = BeautifulSoup(path.read_text(encoding="utf-8"), "html.parser")
        for tag in soup(["script", "style", "nav", "footer", "aside"]):
            tag.decompose()
        for comment in soup.find_all(string=lambda s: isinstance(s, Comment)):
            comment.extract()
        return (soup.find("article") or soup.body).get_text("\n")
    return path.read_text(encoding="utf-8")


def clean_text(text: str, headers: tuple[str, ...] = ()) -> str:
    blocks: list[str] = []

    def keep(m):
        blocks.append(m.group(0))
        return f"\x00CODE{len(blocks) - 1}\x00"

    text = CODE_BLOCK.sub(keep, text).replace("\r\n", "\n")
    text = re.sub("[\u200b\u200c\u200d\ufeff]", "", text)
    text = text.replace("\u3000", " ").replace("\xa0", " ")

    lines = []
    for line in text.split("\n"):
        line = re.sub(r"[ \t]+", " ", line).strip()
        if PAGE_NUMBER.match(line) or any(line.startswith(h) for h in headers):
            continue
        lines.append(HEADING_MARK.sub("", line))

    text = re.sub(r"\n{3,}", "\n\n", "\n".join(lines)).strip()
    for i, block in enumerate(blocks):
        text = text.replace(f"\x00CODE{i}\x00", block)
    return text


docs = pd.DataFrame([
    {
        "doc_id": p.stem,
        "doc_type": p.suffix.lstrip("."),
        "text": clean_text(extract_text(p), headers=("客服问答手册",)),
    }
    for p in sorted((DATA / "docs").iterdir())
    if p.is_file()
])
docs["n_chars"] = docs["text"].str.len()
docs[["doc_id", "doc_type", "n_chars"]]

,doc_id,doc_type,n_chars
0,产品说明,md,708
1,客服问答,txt,383
2,技术博客,html,392
3,转载_客服问答,txt,233


In [4]:
# 看一眼清洗结果是否正常（有没有把代码块压平、有没有残留页码）
sample = docs.loc[docs["doc_id"] == "产品说明", "text"].iloc[0]
print(sample[:400])
print("\n--- 检查 ---")
print("代码块保留:", "```python" in sample)
print("页码残留:", bool(PAGE_NUMBER.search(sample)))
print("零宽字符:", "\u200b" in "".join(docs["text"]))

智学助手产品说明

智学助手是一款面向企业培训场景的问答产品，支持文档导入、知识检索与多轮对话。
本文说明产品能力、接入方式与常见限制。

一、核心能力

- **文档问答**：支持 PDF、Markdown、Word 三类文档导入，自动切分为片段并建立向量索引。
- **多轮对话**：保留最近 10 轮上下文，超出部分自动摘要。
- **引用溯源**：回答中给出的结论会附带原文片段编号，便于人工核对。

二、接入方式

调用方式分为云端 API 与私有化部署两种。云端 API 使用 Bearer Token 鉴权：

```python
import requests

resp = requests.post(
    "https://api.example.com/v1/chat",
    headers={"Authorization": "Bearer <TOKEN>"},


--- 检查 ---
代码块保留: True
页码残留: False
零宽字符: False


## 2. 切分：300 字一段，重叠 60 字

切分参数要在"上下文完整"和"片段命中率"之间权衡：

- 太小：一句话被切断，检索和训练都学不到完整语义；
- 太大：一段里混多个主题，检索不精准；
- 重叠：防止关键句正好落在切口上。

In [5]:
def chunk_text(text: str, size: int = 300, overlap: int = 60) -> list[str]:
    chunks, start = [], 0
    text = text.strip()
    while start < len(text):
        end = min(start + size, len(text))
        if end < len(text):
            window = text[start:end]
            cut = max(window.rfind(p) for p in "。！？\n")
            if cut > size * 0.5:
                end = start + cut + 1
        piece = text[start:end].strip()
        if piece:
            chunks.append(piece)
        if end >= len(text):
            break
        start = max(end - overlap, start + 1)
    return chunks


chunks = pd.DataFrame([
    {"source": row.doc_id, "chunk_id": i, "text": piece}
    for row in docs.itertuples()
    for i, piece in enumerate(chunk_text(row.text))
])
chunks["n_chars"] = chunks["text"].str.len()
print("chunk 总数:", len(chunks))
chunks.groupby("source")["n_chars"].agg(["count", "mean", "max"]).round(0)

chunk 总数: 8


,count,mean,max
source,,,
产品说明,3,275.0,288
客服问答,2,220.0,289
技术博客,2,225.0,277
转载_客服问答,1,233.0,233


## 3. 去重：先精确，再近似

文档里有"转载版"这种近似重复，必须去掉，否则同一段话会在训练集里出现两次、
甚至同时进训练集和验证集。

In [6]:
def norm_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", text).lower()
    return re.sub(r"[^\w\u4e00-\u9fff]", "", text)


def simhash(text: str, bits: int = 64) -> int:
    tokens = re.findall(r"[\u4e00-\u9fff]|[a-zA-Z]+", text)
    vec = np.zeros(bits)
    for token in tokens:
        h = int(hashlib.md5(token.encode("utf-8")).hexdigest(), 16)
        for i in range(bits):
            vec[i] += 1 if (h >> i) & 1 else -1
    return int("".join("1" if v > 0 else "0" for v in vec), 2)


def hamming(a: int, b: int) -> int:
    return bin(a ^ b).count("1")


chunks["simhash"] = chunks["text"].map(simhash)
chunks["exact"] = chunks["text"].map(lambda t: hashlib.md5(norm_text(t).encode("utf-8")).hexdigest())

n0 = len(chunks)
chunks = chunks.drop_duplicates("exact", keep="first").reset_index(drop=True)
n1 = len(chunks)

kept: list[int] = []
for i in range(len(chunks)):
    if all(hamming(chunks["simhash"][i], chunks["simhash"][k]) > 12 for k in kept):
        kept.append(i)
chunks = chunks.iloc[kept].reset_index(drop=True)
n2 = len(chunks)

print(f"去重: {n0} 段 → 精确去重后 {n1} 段（删 {n0 - n1}）→ 近似去重后 {n2} 段（再删 {n1 - n2}）")
chunks.groupby("source").size().to_frame("条数")

去重: 8 段 → 精确去重后 8 段（删 0）→ 近似去重后 7 段（再删 1）


,条数
source,
产品说明,3
客服问答,2
技术博客,2


## 4. 组装两类训练记录

| 类型 | 用途 | 字段 |
| --- | --- | --- |
| `pretrain` | 继续预训练（学领域知识） | `text` |
| `instruction` | 指令微调（学"怎么回答"） | `instruction` / `input` / `output` |

两类数据**不要混在一个文件里**：训练时的 loss 计算方式不同，混在一起会互相干扰。

In [7]:
pretrain = pd.DataFrame({
    "record_id": "doc-" + chunks.index.astype(str).str.zfill(3),
    "kind": "pretrain",
    "source": chunks["source"],
    "text": chunks["text"],
    "prompt": pd.NA,
    "response": pd.NA,
})

# 指令数据来源一：FAQ（嵌套 JSON → 展平）
faq = json.loads((DATA / "raw_sft" / "faq_qa.json").read_text(encoding="utf-8"))
faq_df = pd.json_normalize(faq["faq"], record_path="qa", meta="category").rename(columns={"q": "question", "a": "answer"})
faq_records = pd.DataFrame({
    "record_id": "faq-" + pd.Series(range(1, len(faq_df) + 1)).astype(str),
    "kind": "instruction",
    "source": "faq." + faq_df["category"],
    "text": pd.NA,
    "prompt": faq_df["question"],
    "response": faq_df["answer"],
})

# 指令数据来源二：Alpaca JSONL
alpaca_raw = pd.read_json(DATA / "raw_sft" / "alpaca_raw.jsonl", lines=True)
instruction = alpaca_raw["instruction"].fillna("").astype("string").str.strip()
extra = alpaca_raw["input"].fillna("").astype("string").str.strip()
prompt = np.where(extra.eq(""), instruction, instruction + "\n\n" + extra)
prompt = pd.Series(prompt).str.strip().mask(instruction.eq(""))   # 空 instruction 视为无效样本
alpaca_records = pd.DataFrame({
    "record_id": "alpaca-" + pd.Series(range(1, len(alpaca_raw) + 1)).astype(str),
    "kind": "instruction",
    "source": "alpaca",
    "text": pd.NA,
    "prompt": prompt,
    "response": alpaca_raw["output"],
})

records = pd.concat([pretrain, faq_records, alpaca_records], ignore_index=True)
records["n_chars"] = records["text"].fillna("").str.len() + records["prompt"].fillna("").str.len() + records["response"].fillna("").str.len()
records.groupby(["kind", "source"]).size().to_frame("条数")

条数
kind        source        
instruction alpaca      11
            faq.发票     2
            faq.售后     2
            faq.配送     2
pretrain    产品说明     3
            客服问答     2
            技术博客     2

## 5. 质检：一张报告表交代每一步丢了多少

数据管线最容易出的问题是"悄悄少了 30% 样本却没人发现"。
所以每一步都要记账：**哪条规则、命中多少、剩多少**。

In [8]:
def estimate_tokens(text: str) -> int:
    text = "" if pd.isna(text) else str(text)
    zh = len(re.findall(r"[\u4e00-\u9fff]", text))
    en_words = re.findall(r"[a-zA-Z]+", text)
    en_chars = sum(map(len, en_words))
    rest = max(len(text) - zh - en_chars, 0)
    return int(zh + len(en_words) * 1.3 + rest * 0.25)


records["est_tokens"] = (
    records["text"].map(estimate_tokens)
    + records["prompt"].map(estimate_tokens)
    + records["response"].map(estimate_tokens)
)

checks = pd.DataFrame({
    "规则": ["空内容", "pretrain 太短(<80字)", "instruction 缺 prompt", "instruction 缺 response", "response 过短(<8字)"],
    "命中数": [
        int((records[["text", "prompt", "response"]].fillna("").apply(lambda s: s.str.len()).sum(axis=1) < 10).sum()),
        int(((records["kind"] == "pretrain") & (records["text"].fillna("").str.len() < 80)).sum()),
        int(((records["kind"] == "instruction") & (records["prompt"].fillna("").str.len() < 2)).sum()),
        int(((records["kind"] == "instruction") & (records["response"].isna())).sum()),
        int(((records["kind"] == "instruction") & (records["response"].fillna("").str.len() < 8)).sum()),
    ],
})
checks

,规则,命中数
0,空内容,0
1,pretrain 太短(<80字),0
2,instruction 缺 prompt,1
3,instruction 缺 response,1
4,response 过短(<8字),1


In [9]:
# 应用过滤规则（逐条记录，方便回查）
valid = ~(
    (records["kind"] == "pretrain") & (records["text"].fillna("").str.len() < 80)
)
valid &= ~(
    (records["kind"] == "instruction")
    & ((records["prompt"].fillna("").str.len() < 2) | (records["response"].fillna("").str.len() < 8))
)

print("过滤前:", len(records), "→ 过滤后:", int(valid.sum()))
records = records[valid].reset_index(drop=True)

print("\n估算 token 分布:")
print(records["est_tokens"].describe().round(1).to_string())

过滤前: 24 → 过滤后: 22

估算 token 分布:
count     22.0
mean      69.7
std       66.5
min       12.0
25%       26.2
50%       40.0
75%      103.0
max      236.0


## 6. 划分：按"组"划分，防止泄漏

划分单位取决于数据是否"同源相关"：

- **pretrain**：同一篇文档切出来的 chunk 高度相关，必须整体落在同一边 → 按文档分桶；
- **instruction**：每条样本相互独立（一条问答不影响另一条）→ 按样本分桶。

两种都用**哈希分桶**：稳定、可复现，不需要保存随机状态。

In [10]:
def bucket_of(text: str, mod: int = 100) -> int:
    return int(hashlib.md5(str(text).encode("utf-8")).hexdigest()[:8], 16) % mod


# 文档级：4 篇文档按桶号排序，约 30% 进验证集（至少 1 篇）
doc_bucket = {doc: bucket_of(doc) for doc in docs["doc_id"]}
n_val_docs = max(1, round(len(doc_bucket) * 0.3))
val_docs = {doc for doc, _ in sorted(doc_bucket.items(), key=lambda kv: kv[1])[:n_val_docs]}

# 样本级：指令数据每条独立分桶，桶号 < 30 的进验证集
records["row_bucket"] = records["record_id"].map(bucket_of)

is_val = np.where(
    records["kind"] == "pretrain",
    records["source"].isin(val_docs),
    records["row_bucket"] < 30,
)
records["split"] = np.where(is_val, "val", "train")

print("进验证集的文档:", sorted(val_docs))
pd.crosstab([records["kind"], records["source"]], records["split"], margins=True, margins_name="合计")

进验证集的文档: ['产品说明']


split                 train  val  合计
kind        source                    
instruction alpaca        8    1     9
            faq.发票      1    1     2
            faq.售后      2    0     2
            faq.配送      1    1     2
pretrain    产品说明      0    3     3
            客服问答      2    0     2
            技术博客      2    0     2
合计                     16    6    22

## 7. 导出

三种格式各有用处：

- **JSONL**：训练脚本直接 `json.loads` 逐行读；
- **Parquet**：内部复盘、增量更新；
- **CSV 报告**：给同事看（或用 Excel 打开）。

In [11]:
def write_jsonl(rows: list[dict], path: Path) -> None:
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    print(f"写出 {path.name:28s} {len(rows):>3} 条  {path.stat().st_size:>7,} 字节")


pretrain_rows = records[records["kind"] == "pretrain"][["text"]].to_dict("records")
train_rows = records[(records["kind"] == "instruction") & (records["split"] == "train")]
val_rows = records[(records["kind"] == "instruction") & (records["split"] == "val")]

def to_alpaca(df: pd.DataFrame) -> list[dict]:
    return [{"instruction": p, "input": "", "output": r} for p, r in zip(df["prompt"], df["response"])]


write_jsonl(pretrain_rows, OUT / "pretrain.jsonl")
write_jsonl(to_alpaca(train_rows), OUT / "sft_train.jsonl")
write_jsonl(to_alpaca(val_rows), OUT / "sft_val.jsonl")
records.to_parquet(OUT / "pipeline_records.parquet", index=False)
checks.assign(剩余=len(records)).to_csv(OUT / "quality_report.csv", index=False, encoding="utf-8-sig")

写出 pretrain.jsonl                 7 条    3,925 字节
写出 sft_train.jsonl               12 条    1,776 字节
写出 sft_val.jsonl                  3 条      450 字节


## 8. 验收：回读 + 抽样人工看

交付前必做两件事：

1. **回读**：文件能读、条数对得上、字段齐全；
2. **抽样**：随机抽几条打印出来，人眼读一遍——这一步能发现 90% 的"数据看着能跑但没法用"的问题。

In [12]:
for name in ["pretrain.jsonl", "sft_train.jsonl", "sft_val.jsonl", "pipeline_records.parquet"]:
    path = OUT / name
    print(f"{name:28s} {path.stat().st_size:>8,} 字节")

print("\n回读 pretrain.jsonl 检查:")
lines = (OUT / "pretrain.jsonl").read_text(encoding="utf-8").splitlines()
print("条数:", len(lines), "| 字段:", list(json.loads(lines[0]).keys()))

pretrain.jsonl                  3,925 字节
sft_train.jsonl                 1,776 字节
sft_val.jsonl                     450 字节
pipeline_records.parquet       12,664 字节

回读 pretrain.jsonl 检查:
条数: 7 | 字段: ['text']


In [13]:
# 抽样 2 条指令数据（固定 random_state，方便复现）
for record in train_rows.sample(2, random_state=7).to_dict("records"):
    print(json.dumps({"instruction": record["prompt"], "output": record["response"]}, ensure_ascii=False, indent=2))
    print("-" * 60)

{
  "instruction": "计算毛利率\n\n收入 1000，成本 650",
  "output": "毛利率 = (1000-650)/1000 = 35%。"
}
------------------------------------------------------------
{
  "instruction": "解释 SFT 的含义",
  "output": "SFT 指监督微调（Supervised Fine-Tuning）。"
}
------------------------------------------------------------


## 9. 如果数据是真的：还要补什么

这套代码可以直接用到真实项目，但真实项目还要补五件事：

| 缺口 | 补法 |
| --- | --- |
| PDF / Word 文档 | `pdfplumber`、`python-docx` 抽文本，注意分栏和表格 |
| 数据量很大 | 用 `chunksize` 或按文件并发处理，中间结果存 Parquet |
| 增量更新 | 用 `doc_id + chunk_id + hash` 做幂等，只处理新增/变更文件 |
| 质量把关 | 抽样 100 条人工标注"可用/不可用"，算准确率，再决定阈值 |
| 版本管理 | 每次产出记录：数据源版本、代码 git commit、参数（size/overlap/阈值） |

另外两个容易忽略的点：

- **去重要在划分之前**（本节的顺序是对的），否则训练集和验证集撞车，指标虚高；
- **PII 脱敏**要在导出前做（第 10 节有现成的 `mask_pii`，真实项目一定要接上）。

## 10. 小结

这条管线的每一步都对应前面某一节：

| 步骤 | 用到的技能 | 对应章节 |
| --- | --- | --- |
| 盘点 | `pathlib` + `DataFrame` | 01、02 |
| 抽取 | BeautifulSoup、编码处理 | 02、09 |
| 清洗 | 正则、`.str`、空值处理 | 04、05、09 |
| 切分 | 字符串切片 + 列表推导 | 05、09 |
| 去重 | 哈希 + SimHash | 09 |
| 质检 | `describe`、分组统计 | 03、06 |
| 划分 | 哈希分桶 + `groupby` transform | 06、10 |
| 导出 | JSONL / Parquet / CSV | 02、10 |

**最后一句**：pandas 只是工具，真正决定数据质量的是你**对业务的理解**——
知道什么算重复、什么算噪声、什么算一条好样本。这些只能靠不断抽样、不断回看。